# Intervalos de confianza — Simulación Ad Hoc (Práctica No. 1)

Universidad de los Llanos — Simulación Computacional

Este notebook sigue el procedimiento de la subsección **1.11.1 Statistical
Confidence and Run Length** del Capítulo 1 de [Banks1998] para calcular la
media de la muestra, la desviación estándar de la muestra, la media anchura
(*half-width*) y los intervalos de confianza al 95% y 99% de las cinco
medidas de desempeño, tomando los datos directamente de la hoja **"10
runs"** del archivo `Ejemplo fila banco.xlsx` (10 corridas independientes de
20 clientes).

## 1. Librerías

Se usa `pandas` para leer el Excel, `numpy` para los estadísticos básicos y
`scipy.stats` para el valor crítico de la distribución t de Student.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

## 2. Carga de datos desde el Excel

Se lee directamente la hoja `"10 runs"` del archivo `Ejemplo fila
banco.xlsx` (columnas A:F, 10 filas de datos), en vez de digitar los
valores manualmente.

- Si el notebook se ejecuta en **Google Colab**, se abre un cuadro para
  subir el archivo `Ejemplo fila banco.xlsx`.
- Si se ejecuta de forma **local** (Jupyter), el archivo debe estar en la
  misma carpeta que este notebook (o edite la variable `FILENAME` con la
  ruta correspondiente).

In [ ]:
FILENAME = "Ejemplo fila banco.xlsx"
SHEET = "10 runs"

try:
    from google.colab import files
    if not Path(FILENAME).exists():
        print(f"Sube el archivo '{FILENAME}' (debe contener la hoja '{SHEET}')")
        subido = files.upload()
        FILENAME = next(iter(subido))
except ImportError:
    pass  # no estamos en Colab; se usa el archivo local

runs = pd.read_excel(FILENAME, sheet_name=SHEET, usecols="A:F", nrows=10,
                      index_col="Run")
runs.columns = [c.rstrip(":") for c in runs.columns]  # nombres mas limpios
runs

## 3. Media y desviación estándar de la muestra

Siguiendo la notación de [Banks1998] (Ec. 1.1 y 1.2):

$$\bar{X}(n) = \frac{1}{n}\sum_{i=1}^{n} X_i \qquad\qquad
S^2(n) = \frac{1}{n-1}\sum_{i=1}^{n}\left[X_i - \bar{X}(n)\right]^2$$

donde $n=10$ es el número de corridas (filas leídas del Excel) y $X_i$ es
el valor de la medida de desempeño en la corrida $i$.

In [ ]:
n = len(runs)
media = runs.mean()
std = runs.std(ddof=1)   # ddof=1 -> desviacion estandar MUESTRAL (S)

resumen = pd.DataFrame({"Media (X_bar)": media, "Desv. estandar (S)": std})
resumen

## 4. Valor crítico t y media anchura (half-width)

Para un intervalo de confianza de $(1-\alpha)\times 100\%$, la media anchura
se calcula como (Ec. 1.3 de [Banks1998]):

$$h = t_{n-1,\ 1-\alpha/2}\ \frac{S(n)}{\sqrt{n}}$$

con $n-1$ grados de libertad. Aquí se calcula para $\alpha=0{,}05$ (95% de
confianza) y $\alpha=0{,}01$ (99% de confianza).

In [ ]:
df = n - 1
t_95 = stats.t.ppf(1 - 0.05/2, df)   # t_{n-1, 0.975}
t_99 = stats.t.ppf(1 - 0.01/2, df)   # t_{n-1, 0.995}

hw_95 = t_95 * std / np.sqrt(n)
hw_99 = t_99 * std / np.sqrt(n)

print(f"t (95%, df={df}) = {t_95:.4f}")
print(f"t (99%, df={df}) = {t_99:.4f}")

half_width = pd.DataFrame({"half-width 95%": hw_95, "half-width 99%": hw_99})
half_width

## 5. Intervalos de confianza

El intervalo de confianza para la media poblacional queda definido como
(Ec. 1.4 de [Banks1998]):

$$\bar{X}(n) - h \ \le\ \mu \ \le\ \bar{X}(n) + h$$

In [ ]:
ic = pd.DataFrame({
    "Media": media,
    "IC 95% (inf)": media - hw_95,
    "IC 95% (sup)": media + hw_95,
    "IC 99% (inf)": media - hw_99,
    "IC 99% (sup)": media + hw_99,
})
ic.round(4)

## 6. Interpretación

- Con 95% de confianza, se espera que el intervalo `[IC 95% (inf), IC 95%
  (sup)]` contenga la verdadera media poblacional de cada medida de
  desempeño; con 99% de confianza el intervalo es más ancho, como se
  observa al comparar las columnas anteriores.
- Entre las cinco medidas, **"Average waiting time per customer"** presenta
  el intervalo relativamente más ancho respecto a su media, mientras que
  **"Percent idle time"** es la medida más estable (intervalo más angosto en
  términos relativos).
- Ningún intervalo, ni al 95% ni al 99%, incluye el valor cero, lo cual
  respalda que efectivamente existe espera y ocupación real del servidor en
  el sistema simulado.
- Con solo $n=10$ corridas de 20 clientes, los intervalos son relativamente
  amplios; aumentar el número de corridas y/o el número de clientes por
  corrida (ver Práctica 5.5 con 200 clientes) reduciría la media anchura y,
  por lo tanto, el rango de los intervalos de confianza.

## Referencias

[Banks1998] Jerry Banks. *Handbook of Simulation: Principles, Methodology,
Advances, Applications, and Practice*. John Wiley & Sons, Inc., 1998.
Sección 1.11.1 "Statistical Confidence and Run Length".